<a href="https://colab.research.google.com/github/stacykeago-pergolux/Last-mile-SMK/blob/main/staging_Last_Mile_Table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
/*
.
smk.09.11.26
GEL last-mile carrier shipment staging/dimension model
    source: READ_WH # warehouse
    database: PROD_DB # database
    schema: Z_SOURCE #schema
    source table: PROD_DB.Z_SOURCE.CUSTOM_INGEST_LASTMILE_DELIVERIES # table
    granularity: carrier_ref (or order_id)
    notes:
        columns unpack from `raw` VARIANT column[span_0](start_span)[span_0](end_span)
        empty strings come in as "" - cast to null using nullif()
        timestamps and dates standardized to UTC timestamp_tz
        dest_zip and package_count cast to integer
        ordered by delivered_at asc nulls last, then handover_at asc nulls last
*/

select
    -- core dims & primary identifiers
    nullif(raw:order::varchar, '')                                               as order_id,
    nullif(raw:country::varchar, '')                                             as shop_country_code,
    nullif(raw:carrier::varchar, '')                                             as carrier,
    try_to_number(nullif(raw:carrier_ref::varchar, ''))                          as carrier_ref,
    nullif(raw:service::varchar, '')                                             as service_level,
    nullif(raw:carrier_sla::varchar, '')                                         as carrier_sla,
    nullif(raw:type::varchar, '')                                                as shipment_type,
    nullif(raw:sub_carrier::varchar, '')                                         as sub_carrier,
    nullif(raw:change::varchar, '')                                              as change_status,

    -- destination details
    nullif(raw:dest_city::varchar, '')                                           as dest_city,
    nullif(raw:dest_country::varchar, '')                                        as dest_country,
    nullif(raw:dest_region::varchar, '')                                         as dest_region,
    try_to_number(nullif(raw:dest_zip::varchar, ''))::integer                    as dest_zip,

    -- status & stage
    nullif(raw:stage::varchar, '')                                               as stage,
    nullif(raw:status::varchar, '')                                              as status,
    nullif(raw:on_time::varchar, '')                                             as on_time_status,
    nullif(raw:problem::varchar, '')                                             as problem_description,
    nullif(raw:note::varchar, '')                                                as shipment_note,
    nullif(raw:pod::varchar, '')                                                 as pod_reference,

    -- logistics metrics & volumes
    try_to_number(nullif(raw:packages::varchar, ''))::integer                    as package_count,
    try_to_numeric(nullif(raw:weight_kg::varchar, ''), 10, 2)                    as weight_kg,
    try_to_number(nullif(raw:open_days::varchar, ''))                            as open_days,
    try_to_numeric(nullif(raw:days_handover_to_delivery::varchar, ''), 10, 2)    as days_handover_to_delivery,
    try_to_numeric(nullif(raw:days_hub_to_delivery::varchar, ''), 10, 2)         as days_hub_to_delivery,
    try_to_numeric(nullif(raw:days_vs_planned::varchar, ''), 10, 2)              as days_vs_planned,
    try_to_numeric(nullif(raw:workdays_handover_to_delivery::varchar, ''), 10, 2) as workdays_handover_to_delivery,
    try_to_numeric(nullif(raw:workdays_hub_to_delivery::varchar, ''), 10, 2)     as workdays_hub_to_delivery,

    -- hub details
    nullif(raw:hub::varchar, '')                                                 as hub_code,
    try_to_date(nullif(raw:hub_date::varchar, ''))::timestamp_tz                 as hub_at,

    -- delivery & attempt milestones (actual timestamps)
    coalesce(
        try_to_timestamp_tz(nullif(raw:first_attempt::varchar, '')),
        try_to_timestamp_tz(nullif(raw:first_attempt::varchar, '') || ' +0000'),
        try_to_date(nullif(raw:first_attempt::varchar, ''))::timestamp_tz
    ) as first_attempt_at,

    coalesce(
        try_to_timestamp_tz(nullif(raw:delivered_at::varchar, '')),
        try_to_timestamp_tz(nullif(raw:delivered_at::varchar, '') || ' +0000'),
        try_to_date(nullif(raw:delivered_at::varchar, ''))::timestamp_tz
    ) as delivered_at,

    nullif(raw:delivered_basis::varchar, '')                                     as delivered_basis,

    -- calendar boundaries (converted straight to midnight UTC timestamp)
    try_to_date(nullif(raw:handover_date::varchar, ''))::timestamp_tz             as handover_at,
    try_to_date(nullif(raw:planned_date::varchar, ''))::timestamp_tz              as planned_at,
    nullif(raw:planned_window::varchar, '')                                      as planned_window,
    try_to_date(nullif(raw:run_date::varchar, ''))::timestamp_tz                  as run_at,

    -- ingestion / source metadata
    country                                                                      as source_country,
    carrier                                                                      as source_carrier,
    _source_path::varchar                                                        as _source_path,
    _file_name::varchar                                                          as _file_name,
    _file_created_at                                                             as _file_created_at,
    _loaded_at                                                                   as _loaded_at

from PROD_DB.Z_SOURCE.CUSTOM_INGEST_LASTMILE_DELIVERIES
order by
    delivered_at asc nulls last;
